# 02 — Data Preparation: EN→ID Translation, QC, Final Splits (Fase 1)

**Prerequisites to run this notebook:**
- `torch5050` conda env, CUDA GPU available (used for the live NLLB translation demo).
- `facebook/nllb-200-distilled-600M` already cached locally by HuggingFace (no network needed;
  `HF_HUB_DISABLE_XET=1` and `HF_HUB_OFFLINE=1` are set below before any HF import).
- Data files already present on disk: `data/interim/train_id.jsonl`, `data/interim/test_id.jsonl`,
  `data/processed/captions_{train,val,test}.jsonl`, `data/processed/split_stats.json`
  (produced once by the standalone scripts this notebook documents — see below).

**Scope.** This notebook documents Fase 1's pipeline: `scripts/translate_captions.py` (EN→ID via
NLLB-200), `scripts/quality_check.py` (back-translation QC), `scripts/fix_conclusion.py` (targeted
re-translation fix), and `scripts/build_final_dataset.py` (grouped train/val split).

**Important:** translating the *full* dataset (7,766 train + 2,363 test captioning records, each
split into up to 7 sections) takes on the order of tens of minutes on this GPU and was run **once**
via the standalone scripts above — **not** inside this notebook. What this notebook does instead:
(1) run the real translation function live on a small real sample (5-10 real English captions) to
demonstrate the pipeline actually works, and (2) load and analyze the **already-translated, full**
dataset from `data/processed/` for all statistics and examples.

In [1]:
import os, sys, json, re, time, random
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_OFFLINE", "1")

ROOT = r"C:\Users\Wijdan\Documents\GEMASTIK\Data Mining"
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "scripts"))

import pandas as pd
import matplotlib.pyplot as plt

from scripts.translate_captions import (
    NllbTranslator, parse_sections, load_records, SECTION_HEADERS_EN, SECTION_HEADER_ID,
    apply_terminology_fixes, TERMINOLOGY_FIXES,
)

print("imports OK")

imports OK


## 1. Design: why section-aware translation, not whole-blob (Fase 1, `docs/translation_report.md`)

Ground truth is a fixed 7-section structured report, not a short sentence. Each section
(`DISASTER/BUILDING/ROAD/VEGETATION/WATER_BODY/AGRICULTURE/CONCLUSION` ->
`BENCANA/BANGUNAN/JALAN/VEGETASI/BADAN_AIR/PERTANIAN/KESIMPULAN`) is translated **independently**
and reassembled, rather than translating the whole ~150-300 word blob in one NLLB call. This keeps
each NLLB input short/clean and preserves the report structure exactly — `parse_sections()` below
is the real function that does the split, run live on a real record.

In [2]:
real_train_records = load_records("train", limit=5)
example = real_train_records[0]
sections = parse_sections(example["ground_truth"])
print(f"real record, task={example['task']!r}")
print(f"parsed into {len(sections)} sections:\n")
for header, text in sections:
    print(f"  {header:12s}: {text[:90]}{'...' if len(text) > 90 else ''}")

real record, task='disaster caption'
parsed into 7 sections:

  DISASTER    : Explosion
  BUILDING    : Visible damage to buildings is observed in the post-disaster image, characterized by chang...
  ROAD        : The road network shows no visible obstruction, destruction, or surface alteration in the p...
  VEGETATION  : Localized vegetation loss is evident in the post-disaster image, with reduced spectral ref...
  WATER_BODY  : No water body is visible in either image, and there are no observed alterations.
  AGRICULTURE : No agricultural land or patterns are present in either image, and no changes are detected.
  CONCLUSION  : The explosion caused moderate building damage and localized vegetation loss, primarily con...


## 2. The translator, run live on a small real sample

Loading the real `NllbTranslator` (`facebook/nllb-200-distilled-600M`, fp16 on GPU) and running it
on **5 real English section-texts** pulled straight from `train_release.json` — this proves the
pipeline mechanics work, without paying for the full-dataset translation pass here.

In [3]:
t0 = time.time()
translator = NllbTranslator(batch_size=16)
print(f"loaded in {time.time()-t0:.1f}s on {translator.device}")

real_sections = []
for r in load_records("train", limit=10):
    for header, text in parse_sections(r["ground_truth"]):
        real_sections.append((header, text))
        if len(real_sections) >= 8:
            break
    if len(real_sections) >= 8:
        break

texts = [t for _, t in real_sections]
t0 = time.time()
translated = translator.translate_batch(texts)
print(f"translated {len(texts)} real section-texts in {time.time()-t0:.2f}s\n")

for (header, en), id_ in zip(real_sections, translated):
    print(f"[{header}] EN: {en[:90]}")
    print(f"          ID: {id_[:90]}\n")

[translator] loading facebook/nllb-200-distilled-600M on cuda (fp16=True) ...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

[translator] ready


loaded in 26.0s on cuda


[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


translated 8 real section-texts in 3.04s

[DISASTER] EN: Explosion
          ID: Ledakan

[BUILDING] EN: Visible damage to buildings is observed in the post-disaster image, characterized by chang
          ID: Kerusakan yang terlihat pada bangunan terlihat pada gambar pasca bencana, yang ditandai de

[ROAD] EN: The road network shows no visible obstruction, destruction, or surface alteration in the p
          ID: Jaringan jalan tidak menunjukkan penghalang, kehancuran, atau perubahan permukaan yang ter

[VEGETATION] EN: Localized vegetation loss is evident in the post-disaster image, with reduced spectral ref
          ID: Kehilangan vegetasi lokal terlihat pada gambar pasca bencana, dengan refleksi spektral yan

[WATER_BODY] EN: No water body is visible in either image, and there are no observed alterations.
          ID: Tidak ada benda air yang terlihat dalam kedua gambar, dan tidak ada perubahan yang diamati

[AGRICULTURE] EN: No agricultural land or patterns are present in either

### The terminology fix pass

One known NLLB false-friend, found during manual QC: "foundation" (of a building) machine-translates
to `yayasan` (the Indonesian word for a nonprofit *organisation's* foundation), not `fondasi` (the
structural sense). Applied as a conservative post-translation regex fix — demonstrated live below on
a real sentence containing the false friend.

In [4]:
print("registered terminology fixes:")
for pattern, repl in TERMINOLOGY_FIXES:
    print(f"  {pattern.pattern!r} -> {repl!r}")

demo = "kerusakan pada yayasan bangunan terlihat jelas di gambar pasca bencana"
print(f"\nbefore fix: {demo}")
print(f"after fix : {apply_terminology_fixes(demo)}")

registered terminology fixes:
  '\\byayasan\\b' -> 'fondasi'

before fix: kerusakan pada yayasan bangunan terlihat jelas di gambar pasca bencana
after fix : kerusakan pada fondasi bangunan terlihat jelas di gambar pasca bencana


## 3. Back-translation QC methodology (`scripts/quality_check.py`)

For every translated section, QC back-translates ID->EN and compares against the *original* EN
text with two signals:
- **`similarity`** — `difflib.SequenceMatcher` ratio (0-1), catches wording drift.
- **`length_ratio`** — `len(back-translated) / len(original)` (word count), catches **dropped
  content** specifically.

A record is flagged `needs_review` if any section's `length_ratio < 0.7` or `similarity < 0.45`.
These functions are reused live below (not re-implemented) against a couple of real strings.

In [5]:
from scripts.quality_check import similarity, length_ratio, LENGTH_RATIO_THRESHOLD, SIMILARITY_THRESHOLD

orig = "The explosion caused moderate building damage and localized vegetation loss near the site."
good_back = "The explosion caused moderate damage to buildings and localized vegetation loss near the site."
bad_back  = "The explosion caused moderate building damage."  # trailing content dropped

for label, back in [("good (paraphrase only)", good_back), ("bad (content dropped)", bad_back)]:
    sim = similarity(orig, back)
    lr = length_ratio(orig, back)
    flagged = sim < SIMILARITY_THRESHOLD or lr < LENGTH_RATIO_THRESHOLD
    print(f"{label}:\n  similarity={sim:.3f}  length_ratio={lr:.3f}  needs_review={flagged}\n")

good (paraphrase only):
  similarity=0.902  length_ratio=1.077  needs_review=False

bad (content dropped):
  similarity=0.676  length_ratio=0.462  needs_review=True



## 4. The CONCLUSION-section fix (`scripts/fix_conclusion.py`)

Back-translation QC found the `CONCLUSION` section specifically flagged in 34.3% of train records
(mean `length_ratio` 0.82 vs ~1.0 for other sections) — NLLB's beam search occasionally drops a
trailing sentence on multi-sentence inputs (confirmed not a `max_new_tokens` truncation issue).
**Fix:** split `CONCLUSION` into individual sentences before translating (one level deeper than the
existing 7-section split), translate each independently, then rejoin. Demonstrated live below with
the real `split_sentences()` function on a real multi-sentence CONCLUSION.

In [6]:
from scripts.fix_conclusion import split_sentences

real_conclusion = None
for r in load_records("train", limit=200):
    sections = dict(parse_sections(r["ground_truth"]))
    if "CONCLUSION" in sections and sections["CONCLUSION"].count(".") >= 2:
        real_conclusion = sections["CONCLUSION"]
        break

print("real multi-sentence CONCLUSION text:")
print(real_conclusion)
print("\nsplit into individual sentences (translated independently, then rejoined):")
for s in split_sentences(real_conclusion):
    print(" -", s)

real multi-sentence CONCLUSION text:
The explosion caused severe destruction to central buildings and widespread vegetation loss concentrated in the affected area. The road network and lack of agricultural regions or water bodies demonstrate minimal secondary impacts.

split into individual sentences (translated independently, then rejoined):
 - The explosion caused severe destruction to central buildings and widespread vegetation loss concentrated in the affected area.
 - The road network and lack of agricultural regions or water bodies demonstrate minimal secondary impacts.


## 5. Load the ALREADY-translated full dataset — real stats

Everything from here on loads the real, already-translated, already-split files that the
standalone scripts produced (full run, not reproduced in this notebook).

In [7]:
PROC = os.path.join(ROOT, "data", "processed")
with open(os.path.join(PROC, "split_stats.json"), encoding="utf-8") as f:
    split_stats = json.load(f)
print(json.dumps(split_stats, indent=2))

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f]

train_recs = load_jsonl(os.path.join(PROC, "captions_train.jsonl"))
val_recs   = load_jsonl(os.path.join(PROC, "captions_val.jsonl"))
test_recs  = load_jsonl(os.path.join(PROC, "captions_test.jsonl"))
print(f"\nreal record counts on disk -- train: {len(train_recs)}, val: {len(val_recs)}, test: {len(test_recs)}")
assert len(train_recs) == split_stats["n_train_records"]
assert len(val_recs) == split_stats["n_val_records"]
print("matches split_stats.json exactly.")

{
  "seed": 42,
  "val_fraction_of_groups": 0.1,
  "n_train_records": 6999,
  "n_val_records": 767,
  "n_test_records": 2363,
  "n_train_groups": 6377,
  "n_val_groups": 708
}



real record counts on disk -- train: 6999, val: 767, test: 2363
matches split_stats.json exactly.


## 6. Grouped train/val split — why group by `pre_image_path`

Several records share the same `pre_image_path` with a *different* `post_image_type` (Optical vs
SAR) and near-identical ground truth. A naive per-record random split would leak near-duplicate
examples across train/val. `scripts/build_final_dataset.py` groups by `pre_image_path` before
splitting (seed=42). Verified live below: 0 `pre_image_path` overlap between the real train/val
files on disk.

In [8]:
train_pre = {r["pre_image_path"] for r in train_recs}
val_pre = {r["pre_image_path"] for r in val_recs}
overlap = train_pre & val_pre
print(f"unique pre_image_path -- train: {len(train_pre)}, val: {len(val_pre)}")
print(f"overlap: {len(overlap)} (0 expected -- grouped split)")
assert not overlap

# Show the near-duplicate phenomenon that motivated grouping: find a pre_image_path in train
# that's shared by >1 record (different post_image_type), a real example.
from collections import Counter
counts = Counter(r["pre_image_path"] for r in train_recs)
shared = [(k, v) for k, v in counts.items() if v > 1]
print(f"\n{len(shared)} pre_image_path values shared by >1 train record (this is why grouping matters)")
example_key, n = shared[0]
matches = [r for r in train_recs if r["pre_image_path"] == example_key]
print(f"\nexample: {example_key!r} appears {n} times, post_image_types = "
      f"{[r['post_image_type'] for r in matches]}")

unique pre_image_path -- train: 6377, val: 708
overlap: 0 (0 expected -- grouped split)

622 pre_image_path values shared by >1 train record (this is why grouping matters)

example: 'train_images\\turkey_earthquake0_pre_417.png' appears 2 times, post_image_types = ['Optical', 'SAR']


## 7. Real translated examples from `data/processed/captions_train.jsonl`

In [9]:
rng = random.Random(42)
for r in rng.sample(train_recs, 3):
    print("=" * 90)
    print("EN (excerpt):", r["ground_truth_en"][:200], "...")
    print()
    print("ID (full)   :")
    print(r["ground_truth_id"])
    print()

EN (excerpt): DISASTER: Bushfire
BUILDING: Buildings and surrounding structures in the center of the area appear intact, showing no visible signs of fire damage or structural alteration.
ROAD: No visible disruption ...

ID (full)   :
BENCANA: Kebakaran hutan
BANGUNAN: Bangunan dan struktur sekitar di pusat area tampak utuh, tidak menunjukkan tanda-tanda kerusakan kebakaran atau perubahan struktural yang terlihat.
JALAN: Tidak ada gangguan, pemblokiran, atau pembakaran jalan yang terlihat; pola jalan di sekitar daerah tetap tidak berubah dalam gambar sebelum dan sesudah bencana.
VEGETASI: Vegetasi di sekitar struktur yang dibangun di pusat dan di dalam lanskap yang lebih luas menunjukkan bukti yang substansial dari pembakaran. Perubahan refleksi spektral menunjukkan hilangnya vegetasi, dengan hijau sebelum bencana digantikan oleh daerah yang lebih gelap, terbakar dalam gambar pasca bencana, terutama membentang ke luar dari daerah hutan pusat.
BADAN_AIR: Tidak ada perubahan yang terlihat

## 8. Dataset-wide stats on the final translated corpus (real, computed here)

In [10]:
def section_lengths(records):
    lens = []
    for r in records:
        for line in r["ground_truth_id"].split("\n"):
            if ":" in line:
                header, _, body = line.partition(":")
                lens.append((header.strip(), len(body.split())))
    return pd.DataFrame(lens, columns=["section", "n_words"])

df = section_lengths(train_recs)
print("word-count distribution per section (Indonesian, train split, real data):")
display(df.groupby("section")["n_words"].describe()[["count", "mean", "std", "min", "max"]])

sections_present = [set(re.findall(r"^([A-Z_]+):", r["ground_truth_id"], re.M)) for r in train_recs]
n_with_kesimpulan = sum(1 for s in sections_present if "KESIMPULAN" in s)
print(f"\ntrain records WITH a KESIMPULAN section: {n_with_kesimpulan}/{len(train_recs)} "
      f"({100*n_with_kesimpulan/len(train_recs):.1f}%)")
print(f"train records MISSING KESIMPULAN: {len(train_recs) - n_with_kesimpulan} "
      f"(matches design_decisions.md Fase 2 SS9.2: a genuine ~9% characteristic of the source data)")

word-count distribution per section (Indonesian, train split, real data):


,count,mean,std,min,max
section,,,,,
BADAN_AIR,6961.0,20.280850,7.277247,3.0,90.0
BANGUNAN,6999.0,23.050293,10.445016,7.0,179.0
BENCANA,6999.0,1.416774,0.493060,1.0,2.0
JALAN,6961.0,19.460135,6.662224,7.0,112.0
KESIMPULAN,6324.0,31.206199,5.129589,18.0,54.0
PERTANIAN,6935.0,21.507570,7.231925,7.0,75.0
VEGETASI,6938.0,29.997982,9.607430,8.0,75.0



train records WITH a KESIMPULAN section: 6324/6999 (90.4%)
train records MISSING KESIMPULAN: 675 (matches design_decisions.md Fase 2 SS9.2: a genuine ~9% characteristic of the source data)


## Summary

This notebook re-ran the real translation function (`NllbTranslator.translate_batch`), the real
QC signal functions (`similarity`, `length_ratio`), and the real CONCLUSION-splitting function
(`split_sentences`) live on real data samples, then analyzed the full, already-translated corpus
loaded from `data/processed/`. All numbers (split sizes, section word counts, KESIMPULAN
coverage) were computed from the files on disk in this run. See `docs/translation_report.md` and
`docs/design_decisions.md` §9.2 for the full narrative this notebook is reproducing evidence for.